# Report figure assembly


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Inputs and representative case


In [ ]:
import pandas as pd
from pathology.figures import registration_figures, segmentation_figures, feature_figures, spatial_interaction_figure, modality_figure, copy_spatial_panels
FIGURES = PATHS.output_root / 'report_figures'
FIGURES.mkdir(parents=True, exist_ok=True)
REPRESENTATIVE_CASE = None
summary = pd.read_csv(PATHS.registration / 'registration_summary_v5.csv')
eligible = summary.loc[summary.status.isin(['ok', 'warning'])].sort_values('quality_score', ascending=False)
available = set((path.parent.name for path in (PATHS.run / 'cases').glob('*/features.parquet')))
eligible = eligible.loc[eligible.case_id.isin(available)]
if eligible.empty:
    raise ValueError('No representative case has completed registration, segmentation and features')
case_id = REPRESENTATIVE_CASE or str(eligible.iloc[0].case_id)


## Registration and tissue context


In [ ]:
registration_figures(PATHS, FIGURES, case_id)
segmentation_figures(PATHS, FIGURES, case_id)
feature_figures(PATHS, FIGURES, case_id)


## Statistical panels


In [ ]:
copy_spatial_panels(PATHS, FIGURES)
spatial_interaction_figure(PATHS, FIGURES)
modality_figure(PATHS, FIGURES)
pd.DataFrame({'figure': sorted((path.name for path in FIGURES.glob('*.png')))})
